<a href="https://colab.research.google.com/github/Abhishek-M-K2005/YieldVoyager/blob/main/backend/risk_engine/yieldvoyager3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install xgboost shap pandas numpy scikit-learn requests tqdm

In [2]:
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np
import requests
import shap
import xgboost as xgb
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import StandardScaler


In [3]:
protocols = [
    "aave",
    "curve-dex",
    "makerdao",
    "compound-finance",
    "lido",
    "uniswap",
]

In [4]:
def fetch_protocol(protocol):
   url = f"https://api.llama.fi/protocol/{protocol}"
   response = requests.get(url)
   # if the response is successful, then, extract the json from the response and return the data. else, return empty list.
   if response.ok:
       try:
           data = response.json()
           return data
       except requests.exceptions.JSONDecodeError:
           print(f"Warning: Could not decode JSON for protocol {protocol}. Response content: {response.text}")
           return {}
   else:
       print(f"Warning: Failed to fetch data for protocol {protocol}. Status code: {response.status_code}")
       return {}


In [5]:
def build_dataset(protocol_name, protocol_id):
  data = fetch_protocol(protocol_name)
  rows = []

  listed_at = data.get("listedAt")
  # if the protocol specific Total value locked by date json object is not there, then, return empty dataframe.
  """
  'format like:'
  ........
  "chainTvls":{
        "Ethereum-borrowed":{
          "tvl":[
            {
              "date":1538006400,
              "totalLiquidityUSD":3390
            },
              ......
          ],
          ...
        },
        ....
    },
  ......
  """
  if "chainTvls" not in data:
    return pd.DataFrame()
  for chain in data["chainTvls"]:
    tvl_data = data["chainTvls"][chain].get("tvl", [])
    if len(tvl_data) < 10:
      continue

    current_util_ratio = np.random.uniform(0.20, 0.70)

    for i in range(7, len(tvl_data)):

      latest = tvl_data[i]["totalLiquidityUSD"]
      tvl_24h = tvl_data[i - 1]["totalLiquidityUSD"]
      tvl_7d = tvl_data[i - 7]["totalLiquidityUSD"]

      if tvl_7d == 0 or tvl_24h == 0:
        continue

      tvl_change_24h = (latest - tvl_24h) / tvl_24h
      tvl_change_7d = (latest - tvl_7d) / tvl_7d
      liquidity_depth = latest



      borrowed_key = f"{chain}-borrowed"
      borrowed = data.get("currentChainTvls", {}).get(borrowed_key, 0)
      utilisation_ratio = borrowed / liquidity_depth if liquidity_depth != 0 else 0;

      oracle_price_std = abs(tvl_change_24h) * 10

      liquidation_spike_ratio = abs(tvl_change_24h) * 10

      if listed_at:
        protocol_age_days = int((tvl_data[i]["date"] - listed_at)/86400)
      else:
        protocol_age_days = int((tvl_data[i]["date"] - tvl_data[0]["date"])/86400)

      protocol_age_days = max(1, protocol_age_days)

      audit_count = 1 + (protocol_age_days // 180)
      audit_count = min(5, audit_count)

      util_noise = np.random.normal(0, 0.02)
      current_util_ratio += util_noise;

      current_util_ratio = max(0.05, min(0.95, current_util_ratio))
      utilisation_ratio = current_util_ratio

      base_volatility = np.random.uniform(0.01, 0.05);
      oracle_price_std = base_volatility + (abs(tvl_change_24h) * np.random.uniform(0.5, 1.5))

      if tvl_change_24h < -0.05:
          liquidation_spike_ratio = abs(tvl_change_24h) * np.random.uniform(2.0, 4.0)
      else:
          liquidation_spike_ratio = np.random.uniform(0.0, 0.01)
      instability_label = 1 if tvl_change_7d < -0.30 else 0

      rows.append([
                protocol_id,
                protocol_name,
                chain,
                tvl_data[i]["date"],
                tvl_change_24h,
                tvl_change_7d,
                liquidity_depth,
                utilisation_ratio,
                oracle_price_std,
                liquidation_spike_ratio,
                protocol_age_days,
                audit_count,
                instability_label
            ])
  columns = [
      "protocol_id",
      "protocol_name",
      "chain",
      "date",
      "tvl_change_24h",
      "tvl_change_7d",
      "liquidity_depth",
      "utilisation_ratio",
      "oracle_price_std",
      "liquidation_spike_ratio",
      "protocol_age_days",
      "audit_count",
      "instability_label"
  ]

  return pd.DataFrame(rows, columns=columns )


In [ ]:

all_dfs=[]
for idx, protocol in enumerate(tqdm(protocols)):
  df_p = build_dataset(protocol, idx)
  if not df_p.empty:
    all_dfs.append(df_p)
df = pd.concat(all_dfs, ignore_index=True)
print("Total rows:", len(df))
df

100%|██████████| 6/6 [00:27<00:00,  4.66s/it]

Total rows: 131580


,protocol_id,protocol_name,chain,date,tvl_change_24h,tvl_change_7d,liquidity_depth,utilisation_ratio,oracle_price_std,liquidation_spike_ratio,protocol_age_days,audit_count,instability_label
0,0,aave,Ethereum-staking,1607558400,-0.047319,0.002764,244769550,0.326839,0.088657,0.004764,7,1,0
1,0,aave,Ethereum-staking,1607644800,-0.045431,-0.039229,233649315,0.293184,0.086170,0.006439,8,1,0
2,0,aave,Ethereum-staking,1607731200,-0.100323,-0.160264,210208955,0.257932,0.146804,0.323717,9,1,0
3,0,aave,Ethereum-staking,1607817600,0.125564,-0.067544,236603705,0.258997,0.106234,0.007859,10,1,0
4,0,aave,Ethereum-staking,1607904000,0.024082,-0.053742,242301482,0.269643,0.053166,0.006233,11,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
131575,5,uniswap,Soneium,1773446400,-0.000573,0.019964,2669128,0.761694,0.012681,0.006770,409,3,0
131576,5,uniswap,Soneium,1773532800,0.000244,0.017057,2669780,0.766058,0.017168,0.001378,410,3,0
131577,5,uniswap,Soneium,1773619200,0.004001,0.019683,2680463,0.807817,0.021212,0.006189,411,3,0
131578,5,uniswap,Soneium,1773705600,0.009272,0.027522,2705317,0.787621,0.034331,0.007123,412,3,0


In [ ]:
df.to_csv('data.csv')

In [ ]:
from google.colab import files

files.download('data.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
df1 = df.sort_values(["protocol_id", "date"]).reset_index(drop = True)

In [ ]:
print((df['instability_label'] == 1).sum())

3369


experiment 2


In [6]:


# extract data from defi-llama yields api.
def fetch_structural_risk():
    url = "https://yields.llama.fi/pools"
    response = requests.get(url)
    pools = response.json().get("data", [])

    rows = []
    seen_projects = set()
    for pool in pools:
        project_name = pool.get("project")
        if project_name not in seen_projects:
            rows.append({
                "protocol_name": project_name.lower() if project_name else "",
                "impermanent_loss_risk": 1 if pool.get("ilRisk") == "yes" else 0,
                "is_single_exposure": 1 if pool.get("exposure") == "single" else 0,
            })
            seen_projects.add(project_name)
    return pd.DataFrame(rows)

# build dataset using data.
def build_dataset_2(protocol_name, protocol_id):
    # (Assuming fetch_protocol is already defined in your notebook)
    data = fetch_protocol(protocol_name)
    rows = []


    if "chainTvls" not in data:
      return pd.DataFrame()

    for chain in data["chainTvls"]:
        tvl_data = data["chainTvls"][chain].get("tvl", [])
        if len(tvl_data) < 10: continue

        current_util_ratio = np.random.uniform(0.20, 0.70)

        for i in range(7, len(tvl_data)):
            latest = tvl_data[i]["totalLiquidityUSD"]
            tvl_24h = tvl_data[i - 1]["totalLiquidityUSD"]
            tvl_7d = tvl_data[i - 7]["totalLiquidityUSD"]

            if tvl_7d == 0 or tvl_24h == 0: continue

            tvl_change_24h = (latest - tvl_24h) / tvl_24h
            tvl_change_7d = (latest - tvl_7d) / tvl_7d


            age_days = int((tvl_data[i]["date"] - (tvl_data[0]["date"]))/86400)
            age_days = max(1, age_days)

            # Trying to add non-uniformity in the data to make it a realistic data, which can be learnable by machine learning model.

            current_util_ratio = max(0.05, min(0.95, current_util_ratio + np.random.normal(0, 0.02)))
            oracle_price_std = np.random.uniform(0.01, 0.05) + (abs(tvl_change_24h) * np.random.uniform(0.5, 1.5))
            liquidation_spike = abs(tvl_change_24h) * np.random.uniform(2.0, 4.0) if tvl_change_24h < -0.05 else np.random.uniform(0.0, 0.01)

            # Risk label evaluation method using fixed aggregation.
            tvl_risk = min(40, (max(0, -tvl_change_7d) / 0.30) * 40) # max contribution to risk: 40%
            util_risk = min(30, ((current_util_ratio - 0.60) / 0.40) * 30) if current_util_ratio > 0.60 else 0 # max contribution to risk: 30%
            vol_risk = min(15, (oracle_price_std / 0.20) * 15) # max contribution to risk: 15%
            liq_risk = min(15, (liquidation_spike / 0.10) * 15) # max contribution to risk: 15%

            risk_label = round((tvl_risk + util_risk + vol_risk + liq_risk) / 100.0, 4)

            rows.append([
                protocol_id, protocol_name, chain, tvl_data[i]["date"],
                tvl_change_24h, tvl_change_7d, current_util_ratio,
                oracle_price_std, liquidation_spike, age_days, risk_label
            ])

    columns = ["protocol_id", "protocol_name", "chain", "date", "tvl_change_24h", "tvl_change_7d", "utilisation_ratio", "oracle_price_std", "liquidation_spike_ratio", "protocol_age_days", "risk_label"]
    return pd.DataFrame(rows, columns=columns)

# 3. Execution & Merge
all_dfs = [build_dataset_2(p, i) for i, p in enumerate(tqdm(protocols))]
historical_df = pd.concat(all_dfs, ignore_index=True)
structural_df = fetch_structural_risk()
df = pd.merge(historical_df, structural_df, on="protocol_name", how="left")
df.fillna(0, inplace=True) # Clean up missing structural data

100%|██████████| 6/6 [00:45<00:00,  7.65s/it]


In [7]:

all_dfs=[]
for idx, protocol in enumerate(tqdm(protocols)):
  df_p = build_dataset_2(protocol, idx)
  if not df_p.empty:
    all_dfs.append(df_p)
df = pd.concat(all_dfs, ignore_index=True)
print("Total rows:", len(df))
df

100%|██████████| 6/6 [00:04<00:00,  1.29it/s]

Total rows: 132307


,protocol_id,protocol_name,chain,date,tvl_change_24h,tvl_change_7d,utilisation_ratio,oracle_price_std,liquidation_spike_ratio,protocol_age_days,risk_label
0,0,aave,Ethereum-staking,1607558400,-0.047319,0.002764,0.602283,0.084660,0.009669,7,0.0797
1,0,aave,Ethereum-staking,1607644800,-0.045431,-0.039229,0.612317,0.047893,0.001773,8,0.1001
2,0,aave,Ethereum-staking,1607731200,-0.100323,-0.160264,0.644975,0.106865,0.328974,9,0.4776
3,0,aave,Ethereum-staking,1607817600,0.125564,-0.067544,0.649322,0.186324,0.007692,10,0.2783
4,0,aave,Ethereum-staking,1607904000,0.024082,-0.053742,0.701222,0.034918,0.004538,11,0.1806
...,...,...,...,...,...,...,...,...,...,...,...
132302,5,uniswap,Soneium,1773878400,0.000225,0.023631,0.344257,0.024334,0.002787,414,0.0224
132303,5,uniswap,Soneium,1773964800,0.004185,0.017076,0.323160,0.049000,0.002691,415,0.0408
132304,5,uniswap,Soneium,1774051200,0.001965,0.019660,0.326245,0.034774,0.009232,416,0.0399
132305,5,uniswap,Soneium,1774137600,-0.003595,0.015746,0.324373,0.022091,0.006310,417,0.0260


In [ ]:
df.to_csv('risk.csv')

KeyboardInterrupt: 

In [ ]:
from google.colab import files

files.download('risk.csv')

In [8]:
!pip install tf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.2/60.2 kB 2.8 MB/s eta 0:00:00


In [9]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [10]:
features = [
    "tvl_change_24h",
    "tvl_change_7d",
    "utilisation_ratio",
    "oracle_price_std",
    "liquidation_spike_ratio",
    "protocol_age_days"
]

target = "risk_label"

In [11]:
scaler = StandardScaler()
df[features] = scaler.fit_transform(df[features])

def create_sequences(data, window_size=10):
  X, y = [], []
  for _, group in data.groupby(['protocol_id', 'chain']):
    group_values = group[features].values
    labels = group[target].values

    if len(group_values) > window_size:
      for i in range(window_size, len(group_values)):
        X.append(group_values[i-window_size:i, :])
        y.append(labels[i])
  return np.array(X), np.array(y)

X, y = create_sequences(df, window_size = 10)

In [12]:
import tensorflow as tf
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
from tensorflow.keras.models import Model

In [13]:
def build_rnn(window_size=10, num_features=6):
  inputs = Input(shape=(window_size, num_features))
  lstm_layer = LSTM(32, name="risk_embedding")(inputs)
  x = Dropout(0.2)(lstm_layer)
  x = Dense(16, activation="relu")(x)

  output_score= Dense(1, activation='sigmoid', name="risk_score")(x)
  model = Model(inputs=inputs, outputs=output_score)X_test
  model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["mae"])

  encoder=Model(inputs=inputs, outputs=lstm_layer)

  return model, encoder

model, encoder = build_rnn()
model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 10, 6)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ risk_embedding (LSTM)           │ (None, 32)             │         4,992 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ risk_score (Dense)              │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,537 (21.63 KB)

 Trainable params: 5,537 (21.63 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/20
2617/2617 ━━━━━━━━━━━━━━━━━━━━ 23s 7ms/step - loss: 0.4173 - mae: 0.0766 - val_loss: 0.4077 - val_mae: 0.0683
Epoch 2/20
2617/2617 ━━━━━━━━━━━━━━━━━━━━ 18s 7ms/step - loss: 0.4105 - mae: 0.0684 - val_loss: 0.4074 - val_mae: 0.0670
Epoch 3/20
2617/2617 ━━━━━━━━━━━━━━━━━━━━ 20s 8ms/step - loss: 0.4099 - mae: 0.0675 - val_loss: 0.4070 - val_mae: 0.0668
Epoch 4/20
2617/2617 ━━━━━━━━━━━━━━━━━━━━ 22s 8ms/step - loss: 0.4095 - mae: 0.0669 - val_loss: 0.4068 - val_mae: 0.0669
Epoch 5/20
2617/2617 ━━━━━━━━━━━━━━━━━━━━ 17s 7ms/step - loss: 0.4091 - mae: 0.0665 - val_loss: 0.4063 - val_mae: 0.0648
Epoch 6/20
2617/2617 ━━━━━━━━━━━━━━━━━━━━ 21s 7ms/step - loss: 0.4089 - mae: 0.0661 - val_loss: 0.4062 - val_mae: 0.0645
Epoch 7/20
2617/2617 ━━━━━━━━━━━━━━━━━━━━ 17s 7ms/step - loss: 0.4088 - mae: 0.0659 - val_loss: 0.4061 - val_mae: 0.0642
Epoch 8/20
2617/2617 ━━━━━━━━━━━━━━━━━━━━ 18s 7ms/step - loss: 0.4086 - mae: 0.0657 - val_loss: 0.4066 - val_mae: 0.0620
Epoch 9/20
2617/2617 ━━━━━━━━━━━

In [31]:
model.save('/Users/abhishekmarutikarennavar/Documents/YieldVoyager/YieldVoyager/backend/risk_engine/models/rnn_risk_models.h5')
encoder.save('/Users/abhishekmarutikarennavar/Documents/YieldVoyager/YieldVoyager/backend/risk_engine/models/risk_encoder.h5')

In [36]:
from google.colab import files

files.download('/Users/abhishekmarutikarennavar/Documents/YieldVoyager/YieldVoyager/backend/risk_engine/models/rnn_risk_models.h5')
files.download('/Users/abhishekmarutikarennavar/Documents/YieldVoyager/YieldVoyager/backend/risk_engine/models/risk_encoder.h5')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [32]:
# Test: Get a vector for a single protocol sequence
sample_input = X_test[0:1] # Shape (1, 10, 6)
risk_vector = encoder.predict(sample_input)

print(f"Vector ready for ChromaDB: {risk_vector[0]}")
print(f"Vector Dimension: {len(risk_vector[0])}") # Should be 32

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 182ms/step
Vector ready for ChromaDB: [-0.03244123 -0.04669505 -0.01563296  0.19033529 -0.66745484  0.05900835
  0.21706785  0.05291349 -0.32200143 -0.04633709  0.20068462  0.00306566
  0.1738559  -0.02434564  0.02936996 -0.24733002  0.00752314 -0.07944021
  0.00492224 -0.02239072 -0.12254222 -0.17254968  0.00831275  0.01235946
 -0.03150142 -0.07607295 -0.02929052 -0.07389218  0.00843485 -0.07062065
  0.03162506  0.0426801 ]
Vector Dimension: 32
